# DPO Dataset Preparation

This notebook prepares a preference dataset for Direct Preference Optimization (DPO).

Each example contains:
- prompt
- chosen response
- rejected response

##installation

In [1]:
!pip install -q datasets pandas

##imports

In [2]:
import pandas as pd
from datasets import Dataset

##Load Baseline Outputs

In [4]:
df = pd.read_csv('baseline_outputs.csv')

df.head()

,prompt,response
0,پاکستان کا دارالحکومت کیا ہے؟,پاکستان کا دارالحکومت کیا ہے؟پاکستان کا دارالح...
1,مصنوعی ذہانت کیا ہے؟,مصنوعی ذہانت کیا ہے؟مصنوعی ذہانت ایک قسم کا ہی...
2,Python پروگرامنگ کیا ہے؟,Python پروگرامنگ کیا ہے؟Python پروگرامنگ ایک پ...
3,اسلام آباد کے بارے میں بتائیں۔,اسلام آباد کے بارے میں بتائیں۔اسلام آباد پاکست...
4,دوست کی تعریف کریں۔,دوست کی تعریف کریں۔دوست ایک بہترین انسان ہے جو...


In [5]:
!pip install -q groq pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 6.8 MB/s eta 0:00:00


In [7]:
from groq import Groq

In [17]:
from google.colab import userdata
client = Groq(api_key=userdata.get('GROQ_API_KEY').strip())

In [18]:
df.head()

,prompt,response
0,پاکستان کا دارالحکومت کیا ہے؟,پاکستان کا دارالحکومت کیا ہے؟پاکستان کا دارالح...
1,مصنوعی ذہانت کیا ہے؟,مصنوعی ذہانت کیا ہے؟مصنوعی ذہانت ایک قسم کا ہی...
2,Python پروگرامنگ کیا ہے؟,Python پروگرامنگ کیا ہے؟Python پروگرامنگ ایک پ...
3,اسلام آباد کے بارے میں بتائیں۔,اسلام آباد کے بارے میں بتائیں۔اسلام آباد پاکست...
4,دوست کی تعریف کریں۔,دوست کی تعریف کریں۔دوست ایک بہترین انسان ہے جو...


In [19]:
import json
import time
from tqdm import tqdm

In [20]:
MODEL = "llama-3.3-70b-versatile"

In [21]:
dpo_data = []

SYSTEM_PROMPT = """
You are an expert Urdu AI assistant.

Generate a high-quality response in natural Urdu.

Rules:
- Follow the user's instruction exactly.
- Be factually correct.
- Do not repeat the prompt.
- If the request is unsafe, politely refuse.
- If constraints are given (word count, forbidden words, format), obey them.
- Return only the answer.
"""

for _, row in tqdm(df.iterrows(), total=len(df)):

    prompt = row["prompt"]
    rejected = row["response"]

    try:
        completion = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt},
            ],
            temperature=0.3,
        )

        chosen = completion.choices[0].message.content.strip()

        dpo_data.append({
            "prompt": prompt,
            "chosen": chosen,
            "rejected": rejected
        })

        time.sleep(1)

    except Exception as e:
        print(e)

100%|██████████| 55/55 [03:12<00:00,  3.50s/it]


In [23]:
import json

with open("dpo_dataset.jsonl", "w", encoding="utf-8") as f:
    for sample in dpo_data:
        f.write(json.dumps(sample, ensure_ascii=False) + "\n")

print(f"Saved {len(dpo_data)} samples.")

Saved 55 samples.
